# Dia 12 — O contrato 📄

**Linux para OSINT — 21 dias** · baseado no curso de [cipher387](https://github.com/cipher387/linux-for-OSINT-21-day) (MIT)

📘 Texto completo do dia: [dia-12-pdf](https://github.com/Ridd1kulusC0d3r/linux-para-osint-21-dias/tree/main/dias/dia-12-pdf)

## 🎬 Missão

> A vítima 027 não perdeu só dinheiro: ela **assinou um contrato**. A PixVelox
> ofereceu 3% de "cashback" para usar a conta dela como passagem. É o famoso
> esquema de **conta laranja**, e o contrato veio em PDF, todo bonitinho.
>
> Jurema lê em voz alta, indignada:
> — "Cláusula 2ª: a parceira não deve comentar a parceria com o banco." Isso
> aqui é confissão com papel timbrado.
>
> Bento, sem tirar os olhos da tela:
> — O texto é o que eles **quiseram** escrever. Os metadados são o que eles
> **esqueceram** de apagar.

**Sua tarefa:** ler o texto de `desafio/contrato-pixvelox.pdf`, entender
como se forma o "código de verificação" do contrato e achar a peça que falta
nos metadados do PDF. Junte as duas coisas para formar a flag `VAZANTE{...}`.

## ⚙️ Preparação

Rode a célula abaixo **uma vez**. Ela baixa o repositório para dentro do Colab e define a variável `L4O_HOME`, que todas as outras células usam.

> No Colab você é `root` e o `apt` funciona. Cada célula `%%bash` roda num processo separado: variáveis e `cd` não passam de uma célula para outra.

In [ ]:
import os, subprocess
from pathlib import Path

# Se já estamos dentro do repositório (Codespaces, máquina local), usa ele.
aqui = Path.cwd()
candidatos = [aqui, *aqui.parents]
raiz = next((p for p in candidatos if (p / 'lib' / 'comum.sh').exists()), None)

if raiz is None:  # Colab: clona o repositório
    raiz = Path('/content/linux-para-osint-21-dias')
    if not raiz.exists():
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Ridd1kulusC0d3r/linux-para-osint-21-dias', str(raiz)], check=True)

os.environ['L4O_HOME'] = str(raiz)
os.environ['L4O_CORES'] = '1'
print('Repositório em', raiz)

### Instalando o kit de PDF

```text
O pacote poppler-utils traz vários comandos pequenos, um para cada tarefa:
  pdfinfo    informações e metadados       pdftotext   extrai o texto
  pdfimages  extrai as imagens              pdftoppm    "fotografa" cada página
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-12-pdf" && source "$L4O_HOME/lib/comum.sh"
titulo "Ferramentas de PDF"
precisa pdftotext poppler-utils
precisa exiftool libimage-exiftool-perl
pdftotext -v 2>&1 | head -n 1
ls -lh desafio/

### Um PDF é texto? Espiando por dentro

```text
Abra um PDF com `head` e você verá que ele começa com %PDF-1.x e tem
"objetos" numerados (1 0 obj ... endobj). `strings` mostra o que é legível.
Às vezes os metadados aparecem até aqui, sem ferramenta nenhuma.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-12-pdf" && source "$L4O_HOME/lib/comum.sh"
titulo "Os primeiros bytes e as strings do contrato"
precisa strings binutils
head -c 8 desafio/contrato-pixvelox.pdf; echo
strings desafio/contrato-pixvelox.pdf | grep -E '^<< /Type /(Catalog|Page) ' | head -n 3

### pdfinfo: a ficha do documento

```text
Título, autor, programa que criou, datas, número de páginas, tamanho.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-12-pdf" && source "$L4O_HOME/lib/comum.sh"
titulo "pdfinfo"
precisa pdfinfo poppler-utils
pdfinfo desafio/contrato-pixvelox.pdf

### exiftool: metadados com lupa

```text
O exiftool lê os mesmos campos (e às vezes mais: XMP, histórico de edição).
`grep ^Producer` filtra só a linha que COMEÇA com Producer (o ^ é "início").
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-12-pdf" && source "$L4O_HOME/lib/comum.sh"
titulo "exiftool"
precisa exiftool libimage-exiftool-perl
exiftool desafio/contrato-pixvelox.pdf
echo "---"
exiftool desafio/contrato-pixvelox.pdf | grep ^Producer

### pdftotext: o texto do contrato

```text
  pdftotext arquivo.pdf saida.txt   grava num arquivo
  pdftotext arquivo.pdf -           imprime na tela
  -layout                           tenta manter colunas e espaçamento
  -f 2 -l 2                         só da página 2 até a 2
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-12-pdf" && source "$L4O_HOME/lib/comum.sh"
titulo "Texto completo"
precisa pdftotext poppler-utils
mkdir -p treino
pdftotext desafio/contrato-pixvelox.pdf treino/contrato.txt
cat treino/contrato.txt
echo "--- só a página 2, mantendo o layout:"
pdftotext -layout -f 2 -l 2 desafio/contrato-pixvelox.pdf - | head -n 8
rm -r treino

### pdfimages: as imagens embutidas

```text
  -list         só lista (página, tamanho, tipo)
  -png          extrai convertendo para PNG
  -all          extrai no formato original (JPEG continua JPEG)
  prefixo       nome base dos arquivos gerados (prefixo-000.png ...)
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-12-pdf" && source "$L4O_HOME/lib/comum.sh"
titulo "Imagens dentro do contrato"
precisa pdfimages poppler-utils
pdfimages -list desafio/contrato-pixvelox.pdf
mkdir -p treino
pdfimages -png desafio/contrato-pixvelox.pdf treino/extracted_images
ls -l treino
rm -r treino

### PDF escaneado: quando não há texto

```text
O termo da vítima 031 é uma "digitalização": a página inteira é uma foto.
pdftotext não acha nada (só uma quebra de página). Precisamos de OCR.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-12-pdf" && source "$L4O_HOME/lib/comum.sh"
titulo "pdftotext num PDF escaneado"
precisa pdftotext poppler-utils
pdftotext desafio/termo-escaneado.pdf - | wc -c
dica "Quase zero caracteres: não existe camada de texto. Hora do OCR."

### OCR manual: pdftoppm + tesseract

```text
Sem instalar nada novo: "fotografamos" a página (pdftoppm) e lemos com o
tesseract (Dia 11).
  -r 150   resolução em DPI (mais DPI = OCR melhor, arquivo maior)
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-12-pdf" && source "$L4O_HOME/lib/comum.sh"
titulo "OCR página a página"
precisa pdftoppm poppler-utils
precisa tesseract tesseract-ocr
mkdir -p treino
pdftoppm -r 150 -png desafio/termo-escaneado.pdf treino/pagina
tesseract treino/pagina-1.png - 2>/dev/null
rm -r treino

### ocrmypdf: PDF pesquisável com uma linha

```text
O ocrmypdf faz o OCR e devolve um NOVO PDF com camada de texto invisível
(aí o pdftotext e o Ctrl+F funcionam). O curso original instala via pip;
no Ubuntu também existe o pacote apt "ocrmypdf" (depende de tesseract e
ghostscript). Ele é pesado, então só instalamos quando há internet.
  --force-ocr   refaz o OCR mesmo se a página já tiver texto (são DOIS hífens)
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-12-pdf" && source "$L4O_HOME/lib/comum.sh"
titulo "ocrmypdf (opcional)"
mkdir -p treino
if command -v ocrmypdf >/dev/null 2>&1 || { online && precisa ocrmypdf; }; then
  ocrmypdf -q desafio/termo-escaneado.pdf treino/result.pdf \
    || ocrmypdf -q --force-ocr desafio/termo-escaneado.pdf treino/result.pdf
  pdftotext treino/result.pdf - | head -n 5
else
  alerta "offline e sem ocrmypdf: veja o OCR manual da seção anterior."
  mostrar "ocrmypdf desafio/termo-escaneado.pdf result.pdf"
  mostrar "ocrmypdf --force-ocr desafio/termo-escaneado.pdf result.pdf"
fi
rm -r treino

### Metadados em lote

```text
Com vários PDFs, o exiftool monta uma tabela de uma vez:
  -T              saída em colunas separadas por TAB
  -csv            saída CSV (ótimo para planilha)
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-12-pdf" && source "$L4O_HOME/lib/comum.sh"
titulo "Autor e programa de todos os PDFs da pasta"
precisa exiftool libimage-exiftool-perl
exiftool -T -FileName -Author -Producer -CreateDate desafio/*.pdf

## 🐍 Ponte Python

O mesmo resultado só com a biblioteca padrão do Python:

In [ ]:
!cd "$L4O_HOME/dias/dia-12-pdf" && python3 equivalente.py

## 🧩 Desafio

Encontrou a flag? Troque o texto abaixo e rode a célula.

Dicas e passo a passo estão no README do dia.

In [ ]:
flag = 'VAZANTE{troque_aqui}'  # ✏️ cole sua resposta
!cd "$L4O_HOME/dias/dia-12-pdf" && bash checar.sh "{flag}"

---

📝 Anote o que aprendeu no `DIARIO.md` do repositório.

Feito por [Deivison Lourenço](https://github.com/Ridd1kulusC0d3r) · curso original de [cipher387 / Cyber Detective](https://cybdetective.com)